In [ ]:
import shutil, os

paths_to_clean = [
    "/home/jovyan/work/metastore_db",
    "/home/jovyan/work/setup/spark-warehouse"
]

for path in paths_to_clean:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"Cleaned: {path}")

In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [5]:
spark.sql("create database if not exists spark_db").show()

++
||
++
++



In [6]:
"""
Read offline_students.csv and load the data into offline_students_raw table
"""

offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"

offline_students_raw_df = spark.read.format("csv")\
                                .option("header", True)\
                                .option("quote", "\"")\
                                .option("escape", "\"")\
                                .schema(offline_students_schema)\
                                .load(path = "/home/jovyan/work/data/students_offline.csv")
offline_students_raw_df.show()
offline_students_raw_df.write.mode("overwrite").saveAsTable("spark_db.offline_students_raw")

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{"AddressLine1":"...|[{"Skill":"Apache...|{"email":"xyz@abc...|
|102|    David|  Turner|{"AddressLine1":"...|[{"Skill":"Java",...|{"phone":"9873145...|
|103|    Katie|Mcloskey|{"AddressLine1":"...|[{"Skill":"SQL","...|{"email":"ert89@a...|
|104|   Nasima|  Khatun|{"AddressLine1":"...|[{"Skill":"Hadoop...|{"email":"magt23@...|
|105|   Pritam|    Jain|{"AddressLine1":"...|[{"Skill":"Python...|{"phone":"6984753...|
+---+---------+--------+--------------------+--------------------+--------------------+



In [7]:
spark.sql("select * from spark_db.offline_students_raw").show()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{"AddressLine1":"...|[{"Skill":"Apache...|{"email":"xyz@abc...|
|102|    David|  Turner|{"AddressLine1":"...|[{"Skill":"Java",...|{"phone":"9873145...|
|103|    Katie|Mcloskey|{"AddressLine1":"...|[{"Skill":"SQL","...|{"email":"ert89@a...|
|104|   Nasima|  Khatun|{"AddressLine1":"...|[{"Skill":"Hadoop...|{"email":"magt23@...|
|105|   Pritam|    Jain|{"AddressLine1":"...|[{"Skill":"Python...|{"phone":"6984753...|
+---+---------+--------+--------------------+--------------------+--------------------+



In [ ]:
"""
country wise student count
"""
# {""AddressLine1"":""D104 Gopalan Squire"",""AddressLine2"":""Whitefield"",""City"":""Bangalore"",""Country"":""India"",""Pin"":""560001"",""State"":""Karnataka""}
spark.sql("""
    with offline_students as(
        select id, from_json(address, 
            "struct<
                AddressLine1 string,
                AddressLine2 string,
                City string,
                Country string, 
                Pin string,
                State string
            >"
            ) as address
        from spark_db.offline_students_raw
    )
    select address.country, count(*) as total_count from offline_students group by address.country
""").show() # Here we have defined the schema in the from_json function as a struct type
# This is a very complex query and has a huge performance issue
# Ideally we should not laod the table the way we have we need to parse and load a table that is ready for analysis

+----------------+-----------+
|         country|total_count|
+----------------+-----------+
|           India|          3|
|        Engaland|          1|
|Northern Ireland|          1|
+----------------+-----------+



Spark offers 3 types of complex data types:
1. Struct {}
2. Array []
3. Map 

In [13]:
"""
Prepare an offline_students table that is clean and ready for analysis
"""
# address: # {""AddressLine1"":""D104 Gopalan Squire"",""AddressLine2"":""Whitefield"",""City"":""Bangalore"",""Country"":""India"",""Pin"":""560001"",""State"":""Karnataka""}
address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>" # struct datatype

# skills: [{""Skill"":""Apache Spark"",""YearsOfExperience"":""5""},{""Skill"":""Apache Kafka"",""YearsOfExperience"":""6""}]
skills_schema = "array<struct<Skill string, YearsOfExperience string>>" # array datatype nesting struct datatype

# contact: {""email"":""xyz@abc.com"",""phone"":""9823128923""} / {""phone"":""9873145698""} / {""email"":""magt23@abc.com"",""office"":""7896524689""}
# no consistent fields, hence this would be map. if it was consistent then we would use struct data type
contacts_schema = "map<string, string>" # map data type     # here <string, string> means key and value is string

from pyspark.sql.functions import from_json # type: ignore

offline_students_df = offline_students_raw_df.withColumns({
    "address":  from_json("address", address_schema),
    "skills": from_json("skills", skills_schema),
    "contacts": from_json("contacts", contacts_schema)
})

offline_students_df.show()
offline_students_df.printSchema()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             address|              skills|            contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{D104 Gopalan Squ...|[{Apache Spark, 5...|{email -> xyz@abc...|
|102|    David|  Turner|{109 Park Street,...|[{Java, 12}, {Spr...|{phone -> 9873145...|
|103|    Katie|Mcloskey|{9th Avenue, Dors...|[{SQL, 12}, {PL/S...|{email -> ert89@a...|
|104|   Nasima|  Khatun|{G105 MG Tower, B...|[{Hadoop, 3}, {Ap...|{email -> magt23@...|
|105|   Pritam|    Jain|{M206 Richmond To...|[{Python, 10}, {S...|{phone -> 6984753...|
+---+---------+--------+--------------------+--------------------+--------------------+

root
 |-- ID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- AddressLine1: string (nullable = true)

In [14]:
offline_students_df.write.mode("overwrite").saveAsTable("spark_db.offline_students")

In [16]:
"""
Requirement
Perform the following analysis

    What is country wise student count.
    Find all students with more than 1 years of Spark knowledge
    Find all students who didn't provide phone or whatsapp
"""
# What is country wise student count.
spark.sql("""
    select address.country, count(*) as student_count
    from spark_db.offline_students
    group by address.country
""").show()

+----------------+-------------+
|         country|student_count|
+----------------+-------------+
|           India|            3|
|        Engaland|            1|
|Northern Ireland|            1|
+----------------+-------------+



In [ ]:
# Find all students with more than 1 years of Spark knowledge
spark.sql("""
    with offline_students_skills as(
        select id, firstname, lastname, explode(skills) as skills
        from spark_db.offline_students
    )
    select id, firstname, lastname, skills.skill, skills.yearsofexperience
    from offline_students_skills
    where skills.skill like '%Spark%' and skills.yearsofexperience > 1
""").show()
# explode function created 2 rows with the same keys but the column being exploded will have 1 row for each nested attribute

+---+---------+--------+------------+-----------------+
| id|firstname|lastname|       skill|yearsofexperience|
+---+---------+--------+------------+-----------------+
|101| Prashant|  Pandey|Apache Spark|                5|
|104|   Nasima|  Khatun|Apache Spark|                2|
|105|   Pritam|    Jain|Apache Spark|                3|
+---+---------+--------+------------+-----------------+



In [ ]:
# Find all students who didn't provide phone or whatsapp
spark.sql("""
    select id, firstname, lastname, contacts['email']
    from spark_db.offline_students
    where contacts['phone'] is null and contacts['whatsapp'] is null
""").show()
# here we use the key of contacts to create the query

+---+---------+--------+---------------+
| id|firstname|lastname|contacts[email]|
+---+---------+--------+---------------+
|103|    Katie|Mcloskey|  ert89@abc.com|
|104|   Nasima|  Khatun| magt23@abc.com|
+---+---------+--------+---------------+

